# 01 Data Exploration

Exploratory data analysis for `dataset/data/kelvins_competition_data/train_data.csv`.

Scope constraints: this notebook does not modify the original dataset, does not save a processed dataset, does not train models, and does not create recommendations.

## 1. Load libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 6)

## 2. Load dataset

In [ ]:
DATASET_PATH = Path("../dataset/data/kelvins_competition_data/train_data.csv")

df = pd.read_csv(DATASET_PATH)
df.head()

## 3. Show dataset shape

In [ ]:
rows, columns = df.shape
print(f"Rows: {rows}")
print(f"Columns: {columns}")

## 4. Show first rows

In [ ]:
df.head()

## 5. Show column names

In [ ]:
pd.DataFrame({"column_name": df.columns})

## 6. Show data types

In [ ]:
df.dtypes.to_frame(name="data_type")

## 7. Show missing values summary

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_percentage": df.isna().mean() * 100,
    }
).sort_values("missing_count", ascending=False)

missing_summary

## 8. Show duplicated rows count

In [ ]:
duplicated_rows = df.duplicated().sum()
print(f"Duplicated rows: {duplicated_rows}")

## 9. Show descriptive statistics for key columns

In [ ]:
key_columns = [
    "event_id",
    "time_to_tca",
    "risk",
    "max_risk_estimate",
    "miss_distance",
    "relative_speed",
    "mahalanobis_distance",
]

df[key_columns].describe().T

## Event-level structure analysis

In [ ]:
unique_events = df["event_id"].nunique()
print("Number of unique event_id values:", unique_events)

In [ ]:
rows_per_event = df.groupby("event_id").size().sort_values(ascending=False)

In [ ]:
rows_per_event.describe()

In [ ]:
rows_per_event.head(10)

In [ ]:
non_negative_tca = df[df["time_to_tca"] >= 0].copy()
event_level_df = (
    non_negative_tca
    .sort_values(["event_id", "time_to_tca"])
    .groupby("event_id", as_index=False)
    .first()
)

In [ ]:
event_level_df.shape

In [ ]:
event_level_columns = [
    "event_id",
    "time_to_tca",
    "risk",
    "max_risk_estimate",
    "miss_distance",
    "relative_speed",
    "mahalanobis_distance",
    "mission_id",
    "c_object_type",
]

event_level_df[event_level_columns].head()

## 10. Create derived columns inside the notebook only

In [ ]:
eda_df = df.copy()
eda_df["collision_probability"] = 10 ** eda_df["risk"]
eda_df["max_collision_probability"] = 10 ** eda_df["max_risk_estimate"]

eda_df[["risk", "collision_probability", "max_risk_estimate", "max_collision_probability"]].head()

## 11. Show descriptive statistics for derived probability columns

In [ ]:
probability_columns = ["collision_probability", "max_collision_probability"]

eda_df[probability_columns].describe().T

## 12. Plot basic histograms

In [ ]:
histogram_columns = [
    "risk",
    "max_risk_estimate",
    "miss_distance",
    "relative_speed",
    "time_to_tca",
    "collision_probability",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for ax, column in zip(axes, histogram_columns):
    ax.hist(eda_df[column].dropna(), bins=50, color="#2a6f97", edgecolor="white")
    ax.set_title(column)
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()